# Exercise 9.2 — SHAP values: global and local explanations (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day9/day6-02-shap.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

Exercise 9.1 ranked the variables for the forest as a whole. It said nothing
about *direction* (does a high BMI push the risk up or down?) and nothing about
any one patient. **SHAP values** give both. For every patient and every
variable they say how much that variable moved this patient's predicted
probability away from the average, and they add up exactly to the
prediction. Averaged over patients, they also give a ranking of their own,
which we finish by comparing with every other measure, and with the
coefficients of a logistic regression.

**Theory:** Shapley values from cooperative game theory; the baseline
(expected value); additivity; global summaries (mean absolute SHAP, beeswarm);
local explanations (waterfall plot); coefficients of a logistic regression on
a common scale.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn shap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

The same data ([`data/diabetes_clean.csv`](data/diabetes_clean.csv)), split, predictors
and random forest as Exercise 9.1.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

rf = RandomForestClassifier(n_estimators=500, random_state=1)
rf = rf.fit(train[predictors], train["diabetes"])

test_prob = rf.predict_proba(test[predictors])[:, 1]    # probability of diabetes

------------------------------------------------------------------------

### Part 1 — SHAP values for the test patients

SHAP (SHapley Additive exPlanations) borrows an idea from game theory. Think of
the five predictors as players who together produce a payout: this patient's
predicted probability, minus the average prediction (the **baseline**). The
**Shapley value** of a player is its contribution averaged over every order
in which the players could have joined the team. For a variable, that means:
how much does the prediction change when we learn this patient's value of the
variable, averaged over all the combinations of the other variables we might
already know?

Shapley values have properties that make them easy to read:

- **Additivity (efficiency):** baseline + the SHAP values of all variables = the
  prediction for that patient, exactly.
- **Symmetry:** two variables that contribute the same get the same value.
- **Dummy:** a variable that never changes the prediction gets zero.

A positive SHAP value pushes this patient's risk **up**; a negative one pushes
it **down**.

In Python, the **shap** package's `TreeExplainer` uses the structure of the
trees to compute exact SHAP values quickly, without a background sample: the
"unknown" variables are averaged over the training rows that went down each
branch. We explain the 150 test patients. The result has one set of values
for each class; we keep class 1, diabetes.

In [ ]:
explainer = shap.TreeExplainer(rf)
explanation = explainer(test[predictors])

shap_pos = explanation[:, :, 1]            # SHAP values for class 1 (diabetes)

print("shape:   ", shap_pos.values.shape)  # one row per patient, one column per predictor
print("baseline:", round(shap_pos.base_values[0], 3))   # the average prediction

The baseline is 0.367: the forest's average prediction over the training
rows. (The R tab fills the unknown variables from a sample of 100 training
rows and gets 0.400; the forests differ too, so the two languages' SHAP
values differ a little.)

**Check additivity** for the first test patient: the baseline plus the five
SHAP values should give that patient's predicted probability.

> **Worked answer**

In [ ]:
patient_shap = shap_pos.values[0]
print(np.round(patient_shap, 3))

print(shap_pos.base_values[0] + patient_shap.sum())
print(test_prob[0])

> Both lines give the same number: 0.690 in R, 0.688 in Python. Glucose
> contributes by far the most to this patient's prediction (+0.308 in R,
> +0.350 in Python).

> **Learning points**
>
> - A SHAP value is one variable's contribution to one patient's prediction, measured from the baseline (the average prediction).
> - Baseline + the SHAP values of all variables = the prediction, exactly.
> - Positive values push the predicted risk up, negative values push it down.

------------------------------------------------------------------------

### Part 2 — Global summaries: bar plot and beeswarm

We now have 150 × 5 SHAP values. Two summaries turn them into a picture of the
whole model. The first is the **mean absolute SHAP value** of each variable:
how far, on average, the variable moves a patient's prediction, in either
direction. It is a variable importance, in units of probability.

In [ ]:
shap_importance = pd.Series(np.abs(shap_pos.values).mean(axis=0), index=predictors)

shap_importance.sort_values(ascending=False).round(3)

In [ ]:
shap.plots.bar(shap_pos, show=False)
plt.tight_layout()
plt.show()

On average glucose moves a patient's predicted probability by 0.136;
the other four variables by between 0.028 and 0.064. The
ranking is close to the permutation importance of Exercise 9.1.

The second summary, the **beeswarm**, shows every patient. Each dot is one
patient's SHAP value for one variable; the colour is the patient's value of
that variable (blue low, red high).

In [ ]:
shap.plots.beeswarm(shap_pos, show=False)
plt.tight_layout()
plt.show()

**Question!** Read the glucose row of the beeswarm. Where are the patients with
high glucose, and where are those with low glucose? Is the same true of age?

High glucose sits on the right (pushes the risk up) and low glucose on the
left. Glucose can move a prediction a long way in either direction: from about
−0.3 to about +0.36 in R, and to +0.40 in Python. Age points the same way,
older patients mostly on the right, but more weakly (about ±0.2). Nothing told the forest that
either relation should go one way; it found the direction in the data, and
SHAP makes it visible. This is what a bar of importances cannot show.

> **Learning points**
>
> - The mean absolute SHAP value of a variable is a global importance, measured in units of the prediction.
> - The beeswarm shows, for every patient, how large the contribution is and in which direction, coloured by the variable's value.
> - Unlike impurity or permutation importance, SHAP shows the direction of an effect.

------------------------------------------------------------------------

### Part 3 — One patient: the waterfall plot

A **local** explanation answers: *why did the model give this patient this
risk?* Take the first test patient:

In [ ]:
patient = test.iloc[[0]]                   # double brackets keep it a one-row table
print(patient)
print("predicted probability:", round(test_prob[0], 3))

A 28-year-old with no pregnancies, a BMI of 28.5 and a glucose of 194 mg/dL,
who does **not** have diabetes. The model gives her a probability of 0.69 (0.690 in R, 0.688 in Python).
The waterfall plot starts at the baseline at the bottom and adds one SHAP
value at a time, up to her prediction at the top:

In [ ]:
shap.plots.waterfall(shap_pos[0], show=False)
plt.tight_layout()
plt.show()

Her very high glucose adds about 0.31 (R) or 0.35 (Python) to the baseline; her other
values change little. At a threshold of 0.5 she is a false positive, and the
explanation says why: the model has learned that a glucose of 194 usually means
diabetes. Nothing in the explanation says the model is *right*; it says what
the model did.

The same information can be drawn as a **force plot** (R: `sv_force()`;
Python: `shap.plots.force()`): one horizontal bar with the pushes up and down
facing each other.

**Your turn.** Draw the waterfall for the test patient the model thinks is at
**highest** risk.

> **Worked answer**

In [ ]:
highest = test_prob.argmax()
print(test.iloc[[highest]])

shap.plots.waterfall(shap_pos[highest], show=False)
plt.tight_layout()
plt.show()

> Both forests pick the same patient: 27 years old, no pregnancies, a glucose
> of 165, a BMI of 42.1 and a pedigree function of 0.399, with a predicted
> probability of 0.975 (R) or 0.984 (Python). Every variable pushes her risk
> up; glucose adds about 0.35 (R) or 0.38 (Python), BMI about 0.10. She does
> **not** have diabetes. The model's most confident prediction in the test set
> is wrong, and nothing in the explanation would tell you so.

> **Stuck?**
>
> NumPy arrays have `.argmax()`, the position of the largest value. Then use that
> position to pick the patient out of `shap_pos`.

> **Learning points**
>
> - A local explanation breaks one patient's prediction into contributions that add up from the baseline to the prediction.
> - It explains what the model did for this patient, not whether the model was right.
> - Different patients can be at high risk for different reasons, which a global ranking cannot show.

------------------------------------------------------------------------

### Part 4 — Logistic regression, and all the measures side by side

A logistic regression on the same five predictors is a model we can read
directly: each coefficient is the change in the log-odds of diabetes
for one unit more of that predictor, the others held fixed.

`C=np.inf` switches off scikit-learn's default penalty, so the coefficients
match R's `glm`.

In [ ]:
log_fit = LogisticRegression(C=np.inf, max_iter=5000)
log_fit = log_fit.fit(train[predictors], train["diabetes"])

coefs = pd.DataFrame({"term": predictors, "estimate": log_fit.coef_[0]})
print("intercept:", round(log_fit.intercept_[0], 3))
print(coefs.round(4))

auc_log = roc_auc_score(test["diabetes"], log_fit.predict_proba(test[predictors])[:, 1])
print("test AUC:", round(auc_log, 3))

The logistic regression reaches a test AUC of 0.830, against
0.826 (R) and 0.824 (Python) for the forest: on these five predictors the black box buys
little or nothing. Every coefficient is positive: more of each predictor, more
risk, the same directions the beeswarm showed.

**Question!** The largest coefficient by far is for the pedigree function
(0.590), and the smallest is for age (0.0126). Does that make the pedigree
function the most important predictor?

No. A coefficient is *per unit*, and the units differ: the pedigree function
runs from about 0.1 to 2.4, glucose from about 50 to 200. To put them on a
common footing, multiply each coefficient by the standard deviation of its
predictor in the training rows: the change in log-odds for **one standard
deviation** more.

> **Worked answer**

In [ ]:
coefs["sd"] = train[predictors].std().values
coefs["per_sd"] = coefs["estimate"] * coefs["sd"]

coefs.round(3)

> Per standard deviation: glucose 1.016, BMI 0.464, pregnancies 0.235, pedigree
> 0.220, age 0.149, identical in R and Python (fitting a logistic regression
> involves no randomness). Glucose is by far the strongest predictor; the
> pedigree function, with the largest raw coefficient, comes fourth.

Finally, put the four measures side by side: impurity importance and
permutation importance (Exercise 9.1), mean absolute SHAP, and the size of
the per-SD logistic coefficient. Their units have nothing in common, so divide
each by its own largest value: the top variable of each measure scores 1.

In [ ]:
perm = permutation_importance(rf, test[predictors], test["diabetes"],
                              scoring="roc_auc", n_repeats=20, random_state=1)

comparison = pd.DataFrame({
    "impurity":    rf.feature_importances_,
    "permutation": perm.importances_mean,
    "shap":        shap_importance.values,
    "logistic":    np.abs(coefs["per_sd"].values)}, index=predictors)

comparison = comparison / comparison.max()        # each column divided by its largest value
comparison.round(2)

In [ ]:
ordered = comparison.sort_values("shap")

fig, ax = plt.subplots(figsize=(7, 4))
ordered.plot(kind="barh", ax=ax, width=0.8)
ax.set_xlabel("Importance relative to the top variable of each measure")
ax.legend(title="Measure")
fig.tight_layout()
plt.show()

All four put glucose first by a wide margin. Below it they disagree.
Impurity importance gives BMI, age and the pedigree function around half of
glucose's score or more; permutation importance gives them a quarter or less,
and the pedigree function almost nothing; SHAP lies in between. The logistic
regression agrees that BMI comes second but gives age the least weight of
all. Age and the number of pregnancies are strongly correlated (0.74), and a
model that holds one fixed while moving the other has to share the credit
between them in some way; the forest and the logistic regression share it
differently.

**Question!** You are asked to report "the most important risk factors" in the
model. Which of these measures would you use, and what would you say about the
variables below glucose?

> **Learning points**
>
> - A coefficient is per unit of its predictor; multiply by the predictor's standard deviation before comparing sizes.
> - Impurity importance, permutation importance, mean absolute SHAP and standardised coefficients answer different questions; agreement at the top does not mean agreement below it.
> - Say which measure you used, and what question it answers, whenever you report an importance.

## Stretch

> **Stretch — SHAP values for the logistic regression**
>
> Compute SHAP values for `log_fit` instead of the forest (R: `kernelshap()` with
> the same background; Python: `shap.Explainer(log_fit.predict_proba,
> train[predictors])`). For a model that is a straight line on the log-odds
> scale, what does the beeswarm look like for glucose? Why are the dots for
> glucose ordered by colour so neatly?

> **Stretch — does the background matter?**
>
> In R, rerun `kernelshap()` with a background of 20 patients instead of 100, or
> of 100 patients *with* diabetes only. What happens to the baseline and to the
> SHAP values of the first test patient? What does the baseline now stand for?